# 06-Batch Processing Homework

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 15:06:25 WARN Utils: Your hostname, codespaces-ad5d43, resolves to a loopback address: 127.0.0.1; using 10.0.13.155 instead (on interface eth0)
26/03/08 15:06:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 15:06:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Questions

## 1. Install Spark and PySpark

In [3]:
spark.version

'4.1.1'

## 2. Yellow November 2025

In [5]:
yellow_df = spark.read.parquet('data/yellow/yellow_tripdata_2025-11.parquet')

In [6]:
yellow_df \
    .repartition(4) \
    .write.parquet('data/yellow/repartitioned', mode='overwrite')

In [47]:
!ls -lh data/yellow/repartitioned

total 102M
-rw-r--r-- 1 codespace codespace   0 Mar  8 15:14 _SUCCESS
-rw-r--r-- 1 codespace codespace 26M Mar  8 15:14 part-00000-d9421043-fc78-49c2-a296-47777276431c-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 15:14 part-00001-d9421043-fc78-49c2-a296-47777276431c-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 15:14 part-00002-d9421043-fc78-49c2-a296-47777276431c-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  8 15:14 part-00003-d9421043-fc78-49c2-a296-47777276431c-c000.snappy.parquet


## 3. Count records

In [38]:
yellow_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
yellow_df.createOrReplaceTempView ('yellow_trips')

In [21]:
trips_15th = (
    yellow_df
    .where(
        (F.month(F.col("tpep_pickup_datetime")) == 11) &
        (F.dayofmonth(F.col("tpep_pickup_datetime")) == 15)
    )
)

count_15th = trips_15th.count()
print(f"Yellow trips on 2025-11-15 (pickup on that date): {count_15th}")

[Stage 17:>                                                         (0 + 2) / 2]

Yellow trips on 2025-11-15 (pickup on that date): 162604


In [34]:
spark.sql("""
SELECT 
    COUNT(1)
FROM 
    yellow_trips
WHERE
    TO_DATE(tpep_pickup_datetime) = '2025-11-15' 
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



## 4. Longest trip

In [51]:
spark.sql("""
SELECT
  MAX( (to_unix_timestamp(tpep_dropoff_datetime) - to_unix_timestamp(tpep_pickup_datetime)) / 3600.0 ) AS max_hours
FROM 
    yellow_trips
WHERE 
    tpep_dropoff_datetime IS NOT NULL
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime >= tpep_pickup_datetime;
""").show()

[Stage 50:>                                                         (0 + 2) / 2]

+---------+
|max_hours|
+---------+
|90.646667|
+---------+



## 5. User Interface

In [37]:
spark.sparkContext.uiWebUrl

'http://localhost:4040'

## 6. Least frequent pickup location zone

In [44]:
df_zones = spark.read.parquet('data/zones/')

In [45]:
df_zones.show(n=5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [48]:
yellow_zones = yellow_df.join(df_zones, yellow_df.PULocationID == df_zones.LocationID)

In [50]:
yellow_zones.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = t

In [54]:
yellow_zones.createOrReplaceTempView ('yellow_zones')

In [56]:
spark.sql("""
SELECT Zone, COUNT(*) AS trips
FROM yellow_zones
GROUP BY Zone
ORDER BY trips ASC, Zone ASC
LIMIT 5;
""").show()

[Stage 58:>                                                         (0 + 2) / 2]

+--------------------+-----+
|                Zone|trips|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|         Great Kills|    4|
+--------------------+-----+



In [57]:
spark.stop()